=============================================================================
SEASONAL COMPARISON OD MATRIX HEATMAP
Kitsap Transit — StreetLight Data
=============================================================================
Compare seasonal vs off-season traffic
=============================================================================

In [ ]:
import pandas as pd
import numpy as np
import os 
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

In [ ]:
SUMMER_FILE = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2014360_BI_Lynwood_dest_2024summer\2014360_BI_Lynwood_dest_2024summer_od_trip_all.csv'
WINTER_FILE = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2014361_BI_Lynwood_dest2025winterexcld0425\2014361_BI_Lynwood_dest2025winterexcld0425_od_trip_all.csv'

In [ ]:
OUTPUT_DIR  = r'C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\seasonal_comparison'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
VOLUME_COL   = 'Average Daily O-D Traffic (StL Volume)'
CORRIDOR_COL = 'Destination Zone Name'

In [ ]:
TRANSIT_HOURS = {
    'summer_weekday': {'start': 4.5, 'end': 19.25},
    'summer_sunday':  {'start': 9.0, 'end': 16.0},
    'winter_weekday': {'start': 4.5, 'end': 19.25},
    'winter_sunday':  {'start': 9.0, 'end': 16.0}
}

In [ ]:
def extract_hour(day_part_str):
    try:
        s = str(day_part_str)
        if 'all day' in s.lower():
            return None
        idx = int(s.split(':')[0].strip())
        return idx - 1
    except:
        return None

In [ ]:
def get_day_category(day_type_str):
    s = str(day_type_str).lower()
    if 'sunday' in s:   return 'sunday'
    if 'saturday' in s: return 'saturday'
    return 'weekday'

In [ ]:
# Load both files
df_summer = pd.read_csv(SUMMER_FILE)
df_winter = pd.read_csv(WINTER_FILE)

In [ ]:
for df, label in [(df_summer, 'Summer'), (df_winter, 'Winter')]:
    df['Hour']         = df['Day Part'].apply(extract_hour)
    df['Day_Category'] = df['Day Type'].apply(get_day_category)
    df['Season']       = label

In [ ]:
df_all     = pd.concat([df_summer, df_winter], ignore_index=True)
df_hourly  = df_all[df_all['Hour'].notna()].copy()
corridors  = sorted(df_hourly[CORRIDOR_COL].unique().tolist())

In [ ]:
print(df_hourly['Day_Category'].unique())
print(df_hourly['Day Type'].unique())

In [ ]:
DAY_TYPES  = ['All Days (M-Su)', 'Weekday', 'Saturday', 'Sunday']
day_filter_map = {
    'All Days (M-Su)': None,
    'Weekday':         ['weekday'],
    'Saturday':        ['saturday'],
    'Sunday':          ['sunday']
}

day_filter_map = {
   'All Days (M-Su)': None,
   'Weekday':         ['monday', 'tuesday', 'wednesday', 'thursday', 'friday'],
   'Saturday':        ['saturday'],
   'Sunday':          ['sunday']
}

In [ ]:
colors = {'Summer': '#e67e22', 'Winter': '#2980b9'}

In [ ]:
for corridor in corridors:
    cdata = df_hourly[df_hourly[CORRIDOR_COL] == corridor]
    if cdata.empty:
        continue

    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 10), sharey=False)
    axes = axes.flatten()

    for ax_idx, day_type_label in enumerate(DAY_TYPES):
        ax        = axes[ax_idx]
        day_cats  = day_filter_map[day_type_label]

        for season in ['Summer', 'Winter']:
            sdata = cdata[cdata['Season'] == season]
            if day_cats:
                sdata = sdata[sdata['Day_Category'].isin(day_cats)]
            if sdata.empty:
                continue

            hourly = sdata.groupby('Hour')[VOLUME_COL].mean().reset_index()
            hourly = hourly.sort_values('Hour')

            ax.plot(hourly['Hour'], hourly[VOLUME_COL],
                    color=colors[season], linewidth=2.5,
                    marker='o', markersize=4, label=season)
            ax.fill_between(hourly['Hour'], hourly[VOLUME_COL],
                            alpha=0.12, color=colors[season])

        # Transit service hours shading
        day_cat_key = 'sunday' if 'sunday' in day_type_label.lower() else 'weekday'
        for season_key, shade_color in [('summer', '#e67e22'), ('winter', '#2980b9')]:
            key = f'{season_key}_{day_cat_key}'
            if key in TRANSIT_HOURS:
                svc = TRANSIT_HOURS[key]
                ax.axvspan(svc['start'], svc['end'],
                           alpha=0.06, color=shade_color)

        ax.set_title(day_type_label, fontsize=11, fontweight='bold')
        ax.set_xlabel('Hour of Day', fontsize=9)
        ax.set_ylabel('Avg Daily Vehicle Trips', fontsize=9)
        ax.set_xticks([0, 4, 8, 12, 16, 20, 23])
        ax.set_xticklabels(['12a', '4a', '8a', '12p', '4p', '8p', '11p'], fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9)

        # Annotate peak hours per season
        for season in ['Summer', 'Winter']:
            sdata = cdata[cdata['Season'] == season]
            if day_cats:
                sdata = sdata[sdata['Day_Category'].isin(day_cats)]
            if sdata.empty:
                continue
            hourly = sdata.groupby('Hour')[VOLUME_COL].mean()
            if hourly.empty:
                continue
            peak_hr  = hourly.idxmax()
            peak_val = hourly.max()
            ax.annotate(f'{season}\npeak {int(peak_hr)}:00\n{peak_val:,.0f}',
                        xy=(peak_hr, peak_val),
                        xytext=(8, 6), textcoords='offset points',
                        fontsize=7, color=colors[season],
                        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

    fig.suptitle(
        f'Seasonal Traffic Comparison — {corridor}\n'
        f'Orange = Summer (2024)   Blue = Winter (2025, excl. Apr–May)\n'
        f'Shaded band = Kitsap Transit service hours',
        fontsize=12, y=1.01
    )
    plt.tight_layout()
    outpath = os.path.join(OUTPUT_DIR, f'Seasonal_{corridor}.pdf')
    plt.savefig(outpath, dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  {corridor}: seasonal comparison saved')

=============================================================================
SMALL MULTIPLES — ALL CORRIDORS, SHARED SCALE
=============================================================================

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(16, 20), sharey=False)
axes = axes.flatten()

In [ ]:
# Use All Days filter for the summary view
for idx, corridor in enumerate(corridors):
    ax    = axes[idx]
    cdata = df_hourly[df_hourly[CORRIDOR_COL] == corridor]

    for season in ['Summer', 'Winter']:
        sdata  = cdata[cdata['Season'] == season]
        if sdata.empty:
            continue
        hourly = sdata.groupby('Hour')[VOLUME_COL].mean().reset_index().sort_values('Hour')
        ax.plot(hourly['Hour'], hourly[VOLUME_COL],
                color=colors[season], linewidth=2.5,
                marker='o', markersize=4, label=season)
        ax.fill_between(hourly['Hour'], hourly[VOLUME_COL],
                        alpha=0.12, color=colors[season])

    # Transit shading — weekday as reference
    svc = TRANSIT_HOURS.get('winter_weekday', {})
    if svc:
        ax.axvspan(svc['start'], svc['end'], alpha=0.06, color='green')

    ax.set_title(corridor, fontsize=10, fontweight='bold')
    ax.set_xticks([0, 4, 8, 12, 16, 20, 23])
    ax.set_xticklabels(['12a', '4a', '8a', '12p', '4p', '8p', '11p'], fontsize=8)
    ax.set_ylabel('Avg Daily Trips', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

In [ ]:
for idx in range(len(corridors), len(axes)):
    axes[idx].set_visible(False)

In [ ]:
fig.suptitle(
    'Seasonal Volume Comparison — All Corridors, All Days\n'
    'Orange = Summer 2024   Blue = Winter 2025\n'
    'Green shading = KT weekday service hours',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Seasonal_AllCorridors_Summary.pdf'),
            dpi=300, bbox_inches='tight')
plt.close()
print('  All-corridors summary saved.')

In [ ]:
print('\nDone.')